# Amazon Crawl Pipeline

Notebook crawl dữ liệu bằng **Amazon Scraper API** theo đúng schema trường trong `README.md`.

Nguyên tắc:
1. Không đổi tên field từ API.
2. Không sinh thêm field ngoài schema README.
3. Chỉ bổ sung **1 field danh mục** phục vụ phân tích tập crawl.

In [ ]:
import json
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests

In [ ]:
# 1. Cấu hình
BASE_URL = "https://amazon-scraper-api.omkar.cloud/amazon"
API_KEYS = [
    "ok_b78918dcba75932db4374f100f1202ff",
    "ok_e2b9a96b9ed58c00a44da0ef67d2bfe4",
    "ok_410c1d4cfb5452489479ce75f8b95994",
    "ok_39a4701177daef23d7a3c054a6121d6a",
]
ACTIVE_KEY_IDX = 0
COUNTRY_CODE = "US"

# Danh mục đa dạng
CATEGORY_QUERIES = {
    # Điện tử
    "electronics_laptops": "laptop",
    "electronics_tablets": "tablet",
    "electronics_smartphones": "smartphone",
    "electronics_monitors": "computer monitor",
    "electronics_headphones": "headphones",
    "electronics_keyboards": "mechanical keyboard",
    "electronics_storage_ssd": "external ssd",
    "electronics_networking": "wifi router",
    "electronics_gaming_consoles": "gaming console",

    # Nhà cửa & Nhà bếp
    "home_kitchen_appliances": "kitchen appliances",
    "home_cleaning": "vacuum cleaner",
    "home_air_quality": "air purifier",
    "home_furniture": "office chair",

    # Văn phòng & Trường học
    "office_supplies": "office supplies",
    "office_stationery": "stationery",

    # Thời trang
    "fashion_mens": "men clothing",
    "fashion_womens": "women clothing",
    "fashion_shoes": "running shoes",
    "fashion_bags": "backpack",

    # Làm đẹp & Sức khỏe
    "beauty_skincare": "skincare",
    "beauty_makeup": "makeup",
    "health_personal_care": "personal care",
    "health_supplements": "vitamins supplements",

    # Gia đình
    "baby_products": "baby products",
    "toys_games": "board games",

    # Thể thao & Ngoài trời
    "sports_outdoors": "sports equipment",
    "sports_fitness": "home gym equipment",

    # Thú cưng
    "pet_supplies": "pet supplies",

    # Ô tô & Dụng cụ
    "automotive_accessories": "car accessories",
    "tools_home_improvement": "power tools",

    # Sách
    "books_general": "books",
}

# Cài đặt thu thập: số trang, sắp xếp, mục tiêu số dòng
PAGES_PER_CATEGORY = 35
SORT_BY = "best_sellers"
TARGET_ROWS = 10_000
MAX_PRODUCTS_PER_CATEGORY = None
DEDUPE_BY_ASIN = False

ENRICH_MODE = "sample"  # "none" | "sample" | "full"
ENRICH_SAMPLE_SIZE = 1200
ENRICH_MAX_WORKERS = 8
MAX_TOP_REVIEWS_PER_PRODUCT = 3

REQUEST_TIMEOUT = 35
MIN_SLEEP = 0.03
MAX_SLEEP = 0.08
RETRY = 3

OUT_DIR = Path("data") / "amazon_crawl"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SESSION = requests.Session()


def set_active_api_key(idx):
    global ACTIVE_KEY_IDX
    ACTIVE_KEY_IDX = idx
    SESSION.headers.update({"API-Key": API_KEYS[ACTIVE_KEY_IDX]})


set_active_api_key(ACTIVE_KEY_IDX)

print("Thư mục đầu ra:", OUT_DIR.resolve())
print("Tổng số danh mục:", len(CATEGORY_QUERIES))
print("Số lượng API key:", len(API_KEYS))

Output folder: C:\Users\huynh\Documents\SOS\MATERIAL\TQHDL\GK\LAB01_Data_Visualization\source\data\amazon_crawl
Total category keys: 31
API keys loaded: 4


In [ ]:
# 2. Hàm hỗ trợ
def safe_json_dumps(v):
    try:
        return json.dumps(v, ensure_ascii=False)
    except Exception:
        return None


def request_json(endpoint, params, retries=RETRY, raise_on_final_error=True):
    url = f"{BASE_URL}/{endpoint}"
    last_error = None

    for attempt in range(1, retries + 1):
        try:
            resp = SESSION.get(
                url,
                params=params,
                timeout=REQUEST_TIMEOUT,
            )

            # Success
            if resp.status_code == 200:
                return resp.json()

            # Parse message nếu có
            message = ""
            try:
                message = (resp.json() or {}).get("message", "")
            except Exception:
                message = resp.text or ""
            msg_low = str(message).lower()

            # Hết quota -> tự chuyển sang key tiếp theo
            if "monthly request limit" in msg_low or "exceeded" in msg_low:
                if ACTIVE_KEY_IDX + 1 < len(API_KEYS):
                    new_idx = ACTIVE_KEY_IDX + 1
                    set_active_api_key(new_idx)
                    print(f"[KEY-ROTATE] switched to API key #{new_idx + 1}")
                    if attempt < retries:
                        time.sleep(0.5)
                        continue
                last_error = RuntimeError(
                    f"Quota exceeded on all available keys | endpoint={endpoint} | params={params}"
                )
                break

            # 401/403 thường là key sai hoặc không có quyền
            if resp.status_code in (401, 403):
                last_error = RuntimeError(
                    f"Auth error ({resp.status_code}) | endpoint={endpoint} | params={params}. Check API key."
                )
                # thử key tiếp theo nếu còn
                if ACTIVE_KEY_IDX + 1 < len(API_KEYS):
                    new_idx = ACTIVE_KEY_IDX + 1
                    set_active_api_key(new_idx)
                    print(f"[KEY-ROTATE] switched to API key #{new_idx + 1}")
                    if attempt < retries:
                        time.sleep(0.5)
                        continue
                break

            # Retry nhóm lỗi transient từ phía server / rate limit
            if resp.status_code in (429, 500, 502, 503, 504):
                last_error = RuntimeError(
                    f"Transient HTTP {resp.status_code} | endpoint={endpoint} | params={params}"
                )
                if attempt < retries:
                    time.sleep(1.2 * attempt)
                    continue

            # Các status khác
            last_error = RuntimeError(
                f"HTTP {resp.status_code} | endpoint={endpoint} | params={params} | message={message}"
            )
            if attempt < retries:
                time.sleep(1.0 * attempt)
                continue

        except Exception as e:
            last_error = e
            if attempt < retries:
                time.sleep(1.0 * attempt)
                continue

    # Hết retry
    if raise_on_final_error and last_error is not None:
        raise last_error

    return {}


def search_with_fallback(query, page, country_code, sort_primary=SORT_BY):
    # Gọi search với sort chính, nếu fail thì fallback về relevance để tránh vỡ pipeline.
    # 1) primary sort
    primary_payload = request_json(
        endpoint="search",
        params={
            "query": query,
            "page": page,
            "country_code": country_code,
            "sort_by": sort_primary,
        },
        raise_on_final_error=False,
    )

    if isinstance(primary_payload, dict) and primary_payload.get("results") is not None:
        return primary_payload, sort_primary, None

    # 2) fallback sort
    fallback_sort = "relevance"
    fallback_payload = request_json(
        endpoint="search",
        params={
            "query": query,
            "page": page,
            "country_code": country_code,
            "sort_by": fallback_sort,
        },
        raise_on_final_error=False,
    )

    if isinstance(fallback_payload, dict) and fallback_payload.get("results") is not None:
        msg = f"Primary sort failed. Fallback success: {sort_primary} -> {fallback_sort}"
        return fallback_payload, fallback_sort, msg

    return {}, fallback_sort, f"Search failed on both sort_by={sort_primary} and {fallback_sort}"


def normalize_details_payload(details):
    if not isinstance(details, dict):
        return {}

    fixed = dict(details)

    # Chuẩn hóa các biến thể tương tự README để không vỡ schema khi API thiếu/đổi kiểu
    if not isinstance(fixed.get("main_category"), dict):
        fixed["main_category"] = {}
    if not isinstance(fixed.get("category_hierarchy"), list):
        fixed["category_hierarchy"] = []
    if not isinstance(fixed.get("variation_dimensions"), list):
        fixed["variation_dimensions"] = []
    if not isinstance(fixed.get("variants"), dict):
        fixed["variants"] = {}
    if not isinstance(fixed.get("all_variants"), dict):
        fixed["all_variants"] = {}
    if not isinstance(fixed.get("top_reviews"), list):
        fixed["top_reviews"] = []
    if not isinstance(fixed.get("detailed_rating"), dict):
        fixed["detailed_rating"] = {}

    return fixed


def fill_missing_from_search(row, search_item):
    # Nếu details thiếu, fallback bằng field từ search cùng nghĩa
    if row.get("title") in (None, ""):
        row["title"] = search_item.get("title")
    if row.get("asin") in (None, ""):
        row["asin"] = search_item.get("asin")
    if row.get("link") in (None, ""):
        row["link"] = search_item.get("link")
    if row.get("image_url") in (None, ""):
        row["image_url"] = search_item.get("image_url")

    if row.get("currency") is None:
        row["currency"] = search_item.get("currency")
    if row.get("rating") is None:
        row["rating"] = search_item.get("rating")
    if row.get("reviews") is None:
        row["reviews"] = search_item.get("reviews")
    if row.get("sales_volume") is None:
        row["sales_volume"] = search_item.get("sales_volume")
    if row.get("delivery_info") is None:
        row["delivery_info"] = search_item.get("delivery_info")
    if row.get("number_of_offers") is None:
        row["number_of_offers"] = search_item.get("number_of_offers")
    if row.get("lowest_offer_price") is None:
        row["lowest_offer_price"] = search_item.get("lowest_offer_price")

    if row.get("is_best_seller") is None:
        row["is_best_seller"] = search_item.get("is_best_seller")
    if row.get("is_amazon_choice") is None:
        row["is_amazon_choice"] = search_item.get("is_amazon_choice")
    if row.get("is_prime") is None:
        row["is_prime"] = search_item.get("is_prime")
    if row.get("is_climate_friendly") is None:
        row["is_climate_friendly"] = search_item.get("is_climate_friendly")

    if row.get("current_price") is None and row.get("price") is not None:
        row["current_price"] = row.get("price")
    if row.get("price") is None and row.get("current_price") is not None:
        row["price"] = row.get("current_price")
    if row.get("original_price") is None:
        row["original_price"] = search_item.get("original_price")


def flatten_product_row(category_name, query, search_item, details, top_reviews):
    # Chỉ thêm 1 feature danh mục ngoài schema README
    row = {
        "crawl_category": category_name,
    }

    # Product Search fields (README)
    search_fields = [
        "title",
        "price",
        "original_price",
        "rating",
        "reviews",
        "asin",
        "link",
        "image_url",
        "currency",
        "is_best_seller",
        "is_amazon_choice",
        "is_prime",
        "delivery_info",
        "number_of_offers",
        "lowest_offer_price",
        "has_variations",
        "sales_volume",
        "is_climate_friendly",
    ]
    for f in search_fields:
        row[f] = search_item.get(f)

    # Product Details fields: giữ nguyên tên theo payload + normalize kiểu
    details = normalize_details_payload(details)
    for k, v in details.items():
        row[k] = v

    # Top Reviews endpoint: dùng endpoint reviews nếu có, fallback về details.top_reviews
    if isinstance(top_reviews, list) and len(top_reviews) > 0:
        row["top_reviews"] = top_reviews
    elif "top_reviews" not in row:
        row["top_reviews"] = details.get("top_reviews") or []

    # Rescue miss data do API partial response
    fill_missing_from_search(row, search_item)

    return row

In [4]:
# 3. Thu thập dữ liệu
all_search_items = []
search_errors = []

for category_name, query in CATEGORY_QUERIES.items():
    print(f"\n[SEARCH] category={category_name} | query={query}")
    category_items = []

    for page in range(1, PAGES_PER_CATEGORY + 1):
        payload, used_sort, warn_msg = search_with_fallback(
            query=query,
            page=page,
            country_code=COUNTRY_CODE,
            sort_primary=SORT_BY,
        )

        page_items = payload.get("results", []) if isinstance(payload, dict) else []
        category_items.extend(page_items)

        if warn_msg:
            print(f"  - page={page} -> warning: {warn_msg}")
            search_errors.append(
                {
                    "stage": "search",
                    "category": category_name,
                    "query": query,
                    "page": page,
                    "error": warn_msg,
                }
            )

        print(f"  - page={page} | sort={used_sort} -> {len(page_items)} items")

        # Nếu trang không còn kết quả thì bỏ qua danh mục hiện tại
        if len(page_items) == 0:
            print(f"  - page={page} -> 0 items, skip remaining pages of this category")
            break

        time.sleep(random.uniform(MIN_SLEEP, MAX_SLEEP))

        # Dừng sớm nếu đạt mục tiêu số dòng thô
        if len(all_search_items) + len(category_items) >= TARGET_ROWS:
            break

    if isinstance(MAX_PRODUCTS_PER_CATEGORY, int):
        category_items = category_items[:MAX_PRODUCTS_PER_CATEGORY]

    # Gắn metadata nguồn
    for item in category_items:
        item["_seed_category"] = category_name
        item["_seed_query"] = query

    all_search_items.extend(category_items)

    if len(all_search_items) >= TARGET_ROWS:
        break

# Cắt đúng mục tiêu
all_search_items = all_search_items[:TARGET_ROWS]
print(f"\nTotal raw search items (capped): {len(all_search_items)}")

if DEDUPE_BY_ASIN:
    asin_to_search = {}
    for item in all_search_items:
        asin = item.get("asin")
        if asin and asin not in asin_to_search:
            asin_to_search[asin] = item
    crawl_items = list(asin_to_search.values())
    print(f"Items after dedupe by ASIN: {len(crawl_items)}")
else:
    crawl_items = all_search_items
    print(f"Items without dedupe: {len(crawl_items)}")

rows = []
errors = []

# Chọn tập mẫu để làm giàu dữ liệu (enrich)
if ENRICH_MODE == "full":
    enrich_indices = set(range(len(crawl_items)))
elif ENRICH_MODE == "sample":
    n = min(ENRICH_SAMPLE_SIZE, len(crawl_items))
    enrich_indices = set(random.sample(range(len(crawl_items)), n)) if n > 0 else set()
else:
    enrich_indices = set()

print(f"Enrich mode={ENRICH_MODE} | enrich rows={len(enrich_indices)}")


def enrich_one_item(idx, item):
    asin = item.get("asin")
    category_name = item.get("_seed_category")
    query = item.get("_seed_query")

    try:
        details = {}
        reviews = []

        if idx in enrich_indices and asin:
            details = request_json(
                endpoint="product-details",
                params={"asin": asin, "country_code": COUNTRY_CODE},
                raise_on_final_error=False,
            )

            # Thử lại cho chi tiết sản phẩm nếu dữ liệu rỗng
            if not isinstance(details, dict) or len(details) < 3:
                time.sleep(0.2)
                details = request_json(
                    endpoint="product-details",
                    params={"asin": asin, "country_code": COUNTRY_CODE},
                    raise_on_final_error=False,
                )

            reviews_payload = request_json(
                endpoint="product-reviews/top",
                params={"asin": asin, "country_code": COUNTRY_CODE},
                raise_on_final_error=False,
            )
            reviews = (reviews_payload or {}).get("results", [])[:MAX_TOP_REVIEWS_PER_PRODUCT]

            # Dự phòng nếu thiếu dữ liệu đánh giá
            if (not reviews) and isinstance(details, dict):
                reviews = (details.get("top_reviews") or [])[:MAX_TOP_REVIEWS_PER_PRODUCT]

        row = flatten_product_row(category_name, query, item, details or {}, reviews)
        return idx, row, None
    except Exception as e:
        return idx, None, {"asin": asin, "error": str(e), "category": category_name}


# Xử lý nhanh: làm giàu dữ liệu song song
if ENRICH_MODE == "none":
    for idx, item in enumerate(crawl_items):
        category_name = item.get("_seed_category")
        query = item.get("_seed_query")
        row = flatten_product_row(category_name, query, item, {}, [])
        rows.append(row)
        if (idx + 1) % 500 == 0:
            print(f"Processed {idx + 1}/{len(crawl_items)} items")
else:
    with ThreadPoolExecutor(max_workers=ENRICH_MAX_WORKERS) as executor:
        futures = [
            executor.submit(enrich_one_item, idx, item)
            for idx, item in enumerate(crawl_items)
        ]
        for done_i, future in enumerate(as_completed(futures), start=1):
            _, row, err = future.result()
            if row is not None:
                rows.append(row)
            if err is not None:
                errors.append(err)
            if done_i % 200 == 0:
                print(f"Processed {done_i}/{len(crawl_items)} items")

# Gộp lỗi tìm kiếm vào bảng lỗi chung
if search_errors:
    errors.extend(search_errors)

print(f"\nDone. Success rows: {len(rows)} | Errors: {len(errors)}")


[SEARCH] category=electronics_laptops | query=laptop
  - page=1 | sort=best_sellers -> 16 items
  - page=2 | sort=best_sellers -> 16 items
  - page=3 | sort=best_sellers -> 16 items
  - page=4 | sort=best_sellers -> 16 items
  - page=5 | sort=best_sellers -> 16 items
  - page=6 | sort=best_sellers -> 16 items
  - page=7 | sort=best_sellers -> 16 items
  - page=8 | sort=best_sellers -> 16 items
  - page=9 | sort=best_sellers -> 16 items
  - page=10 | sort=best_sellers -> 16 items
  - page=11 | sort=best_sellers -> 16 items
  - page=12 | sort=best_sellers -> 16 items
  - page=13 | sort=best_sellers -> 16 items
  - page=14 | sort=best_sellers -> 16 items
  - page=15 | sort=best_sellers -> 16 items
  - page=16 | sort=best_sellers -> 16 items
  - page=17 | sort=best_sellers -> 16 items
  - page=18 | sort=best_sellers -> 16 items
  - page=19 | sort=best_sellers -> 16 items
  - page=20 | sort=best_sellers -> 2 items
  - page=21 | sort=best_sellers -> 0 items
  - page=21 -> 0 items, skip rema

In [5]:
# 4. Tạo Dataframe & Kiểm tra chất lượng
df = pd.DataFrame(rows)
err_df = pd.DataFrame(errors)

print("df shape:", df.shape)
print("error shape:", err_df.shape)

if not df.empty:
    # Ưu tiên hiển thị danh mục và các trường cốt lõi
    first_cols = [
        "category",
        "asin",
        "title",
        "price",
        "original_price",
        "rating",
        "reviews",
        "currency",
        "is_best_seller",
        "is_amazon_choice",
        "is_prime",
        "sales_volume",
        "main_category",
        "category_hierarchy",
    ]
    ordered_cols = [c for c in first_cols if c in df.columns] + [c for c in df.columns if c not in first_cols]
    df = df[ordered_cols]

    print("\nNull rate by selected columns:")
    check_cols = [
        c
        for c in [
            "category",
            "asin",
            "title",
            "price",
            "rating",
            "reviews",
            "product_details",
            "top_reviews",
        ]
        if c in df.columns
    ]
    print((df[check_cols].isna().mean() * 100).round(2))

df.head(3)

df shape: (9180, 58)
error shape: (1, 5)

Null rate by selected columns:
asin                0.00
title               0.00
price               2.94
rating              0.25
reviews             0.00
product_details    86.96
top_reviews         0.00
dtype: float64


,asin,title,price,original_price,rating,reviews,currency,is_best_seller,is_amazon_choice,is_prime,...,video_thumbnail,has_video,key_features,full_description,technical_details,product_details,has_aplus_content,aplus_images,has_brand_story,frequently_bought_together
0,B08NHSPCR6,HP Chromebook 11A G8 Education Edition AMD A4-...,NaN,NaN,4.1,1161,None,True,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,B0GR6F8HXV,Apple 2026 MacBook Neo 13-inch Laptop with A18...,749.00,NaN,4.8,73,USD,True,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,B0D7NVS6RV,"Lenovo Yoga 7i 2-in-1 Laptop, 16&quot; 2K Touc...",799.99,NaN,4.5,167,USD,True,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# 5. Xuất dữ liệu
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

csv_path = OUT_DIR / f"amazon_products_{COUNTRY_CODE}_{ts}.csv"
parquet_path = OUT_DIR / f"amazon_products_{COUNTRY_CODE}_{ts}.parquet"
jsonl_path = OUT_DIR / f"amazon_products_{COUNTRY_CODE}_{ts}.jsonl"

err_csv_path = OUT_DIR / f"amazon_errors_{COUNTRY_CODE}_{ts}.csv"

if not df.empty:
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    df.to_parquet(parquet_path, index=False)
    df.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)

if not err_df.empty:
    err_df.to_csv(err_csv_path, index=False, encoding="utf-8-sig")

print("Saved:")
if not df.empty:
    print("-", csv_path.resolve())
    print("-", parquet_path.resolve())
    print("-", jsonl_path.resolve())
if not err_df.empty:
    print("-", err_csv_path.resolve())

Saved:
- C:\Users\huynh\Documents\SOS\MATERIAL\TQHDL\GK\LAB01_Data_Visualization\source\data\amazon_crawl\amazon_products_US_20260324_231228.csv
- C:\Users\huynh\Documents\SOS\MATERIAL\TQHDL\GK\LAB01_Data_Visualization\source\data\amazon_crawl\amazon_products_US_20260324_231228.parquet
- C:\Users\huynh\Documents\SOS\MATERIAL\TQHDL\GK\LAB01_Data_Visualization\source\data\amazon_crawl\amazon_products_US_20260324_231228.jsonl
- C:\Users\huynh\Documents\SOS\MATERIAL\TQHDL\GK\LAB01_Data_Visualization\source\data\amazon_crawl\amazon_errors_US_20260324_231228.csv


In [7]:
# 6. Phân tích nhanh (Tùy chọn)
if not df.empty:
    numeric_cols = [c for c in ["price", "original_price", "current_price", "rating", "reviews"] if c in df.columns]
    if numeric_cols:
        display(df[numeric_cols].describe(include="all").T)

    print("\nRows per crawl category:")
    if "category" in df.columns:
        display(df["crawl_category"].value_counts())

    print("\nSample columns count:", len(df.columns))

,count,mean,std,min,25%,50%,75%,max
price,8910.0,75.408256,159.966252,0.00,13.99,26.215,69.99,3399.99
original_price,4096.0,84.605107,155.758506,2.85,19.99,34.990,89.99,2499.99
current_price,8910.0,75.407519,159.966087,0.00,13.99,26.215,69.99,3399.99
rating,9157.0,4.448247,0.295895,1.00,4.30,4.500,4.60,5.00
reviews,9180.0,9543.669826,22589.568778,0.00,405.00,2158.500,8563.00,659066.00



Rows per crawl category:

Sample columns count: 58
